# Particion Temporal y Normalizacion con metodo por Transecto y metodo General

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 5 (adaptado a transectos): Partición temporal y normalización.
Entrada: ventanas generadas en Código 4 (carpeta windows/)
Salida: conjuntos particionados y normalizados en windows_partitioned/
Estructura:
    windows_partitioned/
        by_transect/
            ml/
                Transecto_1/
                    train_X.npy, train_y.npy, val_X.npy, val_y.npy, test_X.npy, test_y.npy
                    features.json (opcional)
                Transecto_2/ ...
            dl/
                Transecto_1/
                    ... (datos normalizados) + scaler_X.pkl, scaler_y.pkl
        global/
            ml/
                Estacion_1/ ...
            dl/
                Estacion_1/ ...
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
WINDOWS_DIR = os.path.join(BASE_DIR, "windows")
OUTPUT_DIR = os.path.join(BASE_DIR, "windows_partitioned")

# Subcarpetas de entrada
INPUT_ML_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "ml")
INPUT_DL_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "dl")
INPUT_ML_GLOBAL = os.path.join(WINDOWS_DIR, "global", "ml")
INPUT_DL_GLOBAL = os.path.join(WINDOWS_DIR, "global", "dl")

# Subcarpetas de salida
OUTPUT_ML_TRANSECT = os.path.join(OUTPUT_DIR, "by_transect", "ml")
OUTPUT_DL_TRANSECT = os.path.join(OUTPUT_DIR, "by_transect", "dl")
OUTPUT_ML_GLOBAL = os.path.join(OUTPUT_DIR, "global", "ml")
OUTPUT_DL_GLOBAL = os.path.join(OUTPUT_DIR, "global", "dl")

for path in [OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

# Parámetros
WINDOW_IN = 72
WINDOW_OUT = 72
TARGET_COL = "O3"

# Fechas de corte
TRAIN_START = pd.to_datetime("2010-01-01 00:00:00")
TRAIN_END   = pd.to_datetime("2023-12-31 23:00:00")
VAL_START   = pd.to_datetime("2024-01-01 00:00:00")
VAL_END     = pd.to_datetime("2024-12-31 23:00:00")
TEST_START  = pd.to_datetime("2025-01-01 00:00:00")

# Márgenes para evitar fuga temporal.
# La ventana usa WINDOW_IN horas de entrada y WINDOW_OUT horas de salida.
INPUT_MARGIN = pd.Timedelta(hours=WINDOW_IN)
OUTPUT_MARGIN = pd.Timedelta(hours=WINDOW_OUT - 1)

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def load_entity_data(entity_dir, entity_name):
    X_path = os.path.join(entity_dir, f"{entity_name}_X.npy")
    y_path = os.path.join(entity_dir, f"{entity_name}_y.npy")
    ts_path = os.path.join(entity_dir, f"{entity_name}_timestamps.npy")

    if not (os.path.exists(X_path) and os.path.exists(y_path) and os.path.exists(ts_path)):
        print(f"    Faltan archivos para {entity_name} en {entity_dir}")
        return None, None, None

    X = np.load(X_path)
    y = np.load(y_path)
    timestamps = pd.to_datetime(np.load(ts_path))

    return X, y, timestamps


def split_by_timestamps(X, y, timestamps):
    """
    Divide usando la marca temporal de anclaje de la ventana, aplicando margen
    para que ni la entrada ni la salida de cada ventana crucen los cortes.
    """
    train_mask = (
        (timestamps >= (TRAIN_START + INPUT_MARGIN)) &
        (timestamps <= (TRAIN_END - OUTPUT_MARGIN))
    )

    val_mask = (
        (timestamps >= (VAL_START + INPUT_MARGIN)) &
        (timestamps <= (VAL_END - OUTPUT_MARGIN))
    )

    test_mask = timestamps >= (TEST_START + INPUT_MARGIN)

    # Verificaciones de solapamiento
    assert not (train_mask & val_mask).any(), "Solapamiento train/val"
    assert not (train_mask & test_mask).any(), "Solapamiento train/test"
    assert not (val_mask & test_mask).any(), "Solapamiento val/test"

    X_train, y_train = X[train_mask], y[train_mask]
    X_val, y_val     = X[val_mask], y[val_mask]
    X_test, y_test   = X[test_mask], y[test_mask]

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)


def _scale_X_split(scaler_X, X_split, win_in, n_feat, fit=False):
    """
    Escala un bloque 3D. Si está vacío, devuelve un array vacío con la misma
    estructura de salida.
    """
    if X_split.shape[0] == 0:
        return np.empty((0, win_in, n_feat), dtype=np.float32)

    X_flat = X_split.reshape(-1, n_feat)
    if fit:
        X_scaled = scaler_X.fit_transform(X_flat)
    else:
        X_scaled = scaler_X.transform(X_flat)

    return X_scaled.reshape(X_split.shape[0], win_in, n_feat).astype(np.float32)


def _scale_y_split(scaler_y, y_split, fit=False):
    """
    Escala un bloque 2D de salida. Si está vacío, devuelve un array vacío
    con forma (0, WINDOW_OUT).
    """
    if y_split.shape[0] == 0:
        return np.empty((0, WINDOW_OUT), dtype=np.float32)

    y_flat = y_split.reshape(-1, 1)
    if fit:
        y_scaled = scaler_y.fit_transform(y_flat)
    else:
        y_scaled = scaler_y.transform(y_flat)

    return y_scaled.reshape(y_split.shape[0], WINDOW_OUT).astype(np.float32)


def normalize_data(X_train, X_val, X_test, y_train, y_val, y_test):
    """
    Normaliza X e y usando solo train para ajustar los escaladores.
    Permite val y test vacíos sin fallar.
    """
    n_train, win_in, n_feat = X_train.shape

    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_train_sc = _scale_X_split(scaler_X, X_train, win_in, n_feat, fit=True)
    X_val_sc   = _scale_X_split(scaler_X, X_val, win_in, n_feat, fit=False)
    X_test_sc  = _scale_X_split(scaler_X, X_test, win_in, n_feat, fit=False)

    y_train_sc = _scale_y_split(scaler_y, y_train, fit=True)
    y_val_sc   = _scale_y_split(scaler_y, y_val, fit=False)
    y_test_sc  = _scale_y_split(scaler_y, y_test, fit=False)

    return X_train_sc, X_val_sc, X_test_sc, y_train_sc, y_val_sc, y_test_sc, scaler_X, scaler_y


def save_split_ml(output_dir, entity_name, X_train, y_train, X_val, y_val, X_test, y_test):
    save_dir = os.path.join(output_dir, entity_name)
    os.makedirs(save_dir, exist_ok=True)

    np.save(os.path.join(save_dir, "train_X.npy"), X_train)
    np.save(os.path.join(save_dir, "train_y.npy"), y_train)
    np.save(os.path.join(save_dir, "val_X.npy"), X_val)
    np.save(os.path.join(save_dir, "val_y.npy"), y_val)
    np.save(os.path.join(save_dir, "test_X.npy"), X_test)
    np.save(os.path.join(save_dir, "test_y.npy"), y_test)


def save_split_dl(output_dir, entity_name, X_train, y_train, X_val, y_val, X_test, y_test, scaler_X, scaler_y):
    save_dir = os.path.join(output_dir, entity_name)
    os.makedirs(save_dir, exist_ok=True)

    np.save(os.path.join(save_dir, "train_X.npy"), X_train)
    np.save(os.path.join(save_dir, "train_y.npy"), y_train)
    np.save(os.path.join(save_dir, "val_X.npy"), X_val)
    np.save(os.path.join(save_dir, "val_y.npy"), y_val)
    np.save(os.path.join(save_dir, "test_X.npy"), X_test)
    np.save(os.path.join(save_dir, "test_y.npy"), y_test)

    with open(os.path.join(save_dir, "scaler_X.pkl"), "wb") as f:
        pickle.dump(scaler_X, f)

    with open(os.path.join(save_dir, "scaler_y.pkl"), "wb") as f:
        pickle.dump(scaler_y, f)


def process_entity(entity_name, input_ml_dir, input_dl_dir, output_ml_dir, output_dl_dir):
    print(f"  Procesando {entity_name}...")

    X_ml, y_ml, ts_ml = load_entity_data(input_ml_dir, entity_name)
    if X_ml is None:
        return

    X_dl, y_dl, ts_dl = load_entity_data(input_dl_dir, entity_name)
    if X_dl is None:
        return

    if not np.array_equal(ts_ml, ts_dl):
        print(f"    Error: timestamps no coinciden en {entity_name}")
        return

    (X_train_ml, y_train_ml), (X_val_ml, y_val_ml), (X_test_ml, y_test_ml) = split_by_timestamps(X_ml, y_ml, ts_ml)
    (X_train_dl, y_train_dl), (X_val_dl, y_val_dl), (X_test_dl, y_test_dl) = split_by_timestamps(X_dl, y_dl, ts_dl)

    if len(X_train_ml) == 0:
        print(f"    Sin datos de train en {entity_name}")
        return

    save_split_ml(
        output_ml_dir, entity_name,
        X_train_ml, y_train_ml,
        X_val_ml, y_val_ml,
        X_test_ml, y_test_ml
    )

    data = normalize_data(
        X_train_dl, X_val_dl, X_test_dl,
        y_train_dl, y_val_dl, y_test_dl
    )

    save_split_dl(output_dl_dir, entity_name, *data)


# ============================================================================
# PROCESAMIENTO
# ============================================================================

def process_by_transect():
    print("\n--- Procesando por transecto ---")

    ml_files = [f.stem.replace("_X", "") for f in Path(INPUT_ML_TRANSECT).glob("*_X.npy")]
    dl_files = [f.stem.replace("_X", "") for f in Path(INPUT_DL_TRANSECT).glob("*_X.npy")]

    for entity in sorted(set(ml_files) & set(dl_files)):
        process_entity(
            entity,
            INPUT_ML_TRANSECT,
            INPUT_DL_TRANSECT,
            OUTPUT_ML_TRANSECT,
            OUTPUT_DL_TRANSECT
        )


def process_global():
    print("\n--- Procesando global ---")

    ml_files = [f.stem.replace("_X", "") for f in Path(INPUT_ML_GLOBAL).glob("*_X.npy")]
    dl_files = [f.stem.replace("_X", "") for f in Path(INPUT_DL_GLOBAL).glob("*_X.npy")]

    for entity in sorted(set(ml_files) & set(dl_files)):
        process_entity(
            entity,
            INPUT_ML_GLOBAL,
            INPUT_DL_GLOBAL,
            OUTPUT_ML_GLOBAL,
            OUTPUT_DL_GLOBAL
        )


# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    print("=" * 60)
    print("Partición temporal + normalización")
    print("=" * 60)

    process_by_transect()
    process_global()

    print("\nProceso completado en:")
    print(OUTPUT_DIR)

Partición temporal + normalización

--- Procesando por transecto ---
  Procesando Transecto_1...
    Sin datos de train en Transecto_1
  Procesando Transecto_2...
    Sin datos de train en Transecto_2

--- Procesando global ---
  Procesando T1_E1_Alicante...
    Sin datos de train en T1_E1_Alicante
  Procesando T1_E2_Elda...
    Sin datos de train en T1_E2_Elda
  Procesando T2_E1_Elche...
    Sin datos de train en T2_E1_Elche
  Procesando T2_E2_Elda...
    Sin datos de train en T2_E2_Elda

Proceso completado en:
/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/windows_partitioned
